# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem]


O **PROJETO PRÁTICO** deve ser feito utilizando o **Google Colab** com uma conta sua vinculada ao Gmail. O link do seu notebook armazenado no Google Drive e o link de um repositório no GitHub devem ser enviados usando o seguinte formulário:

> https://forms.gle/D4gLqP1iGgyn2hbH8


**IMPORTANTE**: A submissão deve ser feita até o dia **04/12 (sexta-feira)** APENAS POR UM INTEGRANTE DA EQUIPE, até às 23h59. Por favor, lembre-se de dar permissão de ACESSO IRRESTRITO para o professor da disciplina.

### **EQUIPE**

---

**Integrante 01:**

Guilherme Araujo dos Santos — RA: 11202020294

**Integrante 02:**

Ian Victor Toniolo Silva — RA: 11202020351

### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

Cada equipe deve selecionar um Grande Modelo de Linguagem (*Large Language Model - LMM*).



Por favor, informe os dados do LLM selecionado:

**LLM**: Claude Haiku 5.5 (Anthropic), identificador na API: `claude-haiku-5-5`

**Link para a documentação oficial**: https://docs.claude.com/en/docs/about-claude/models/overview

### **API (Opcional)**
---

Por favor, informe os dados da API selecionada:

**API**: MediaWiki Action API (Wikipédia em português)

**Site oficial**: https://www.mediawiki.org

**Link para a documentação oficial**: https://www.mediawiki.org/wiki/API:Main_page

### **DESCRIÇÃO**
---

Implementar um `notebook` no `Google Colab` que faça uso do framework **`LangChain`** (obrigatório) e de um **LLM** aplicando, no mínimo, DUAS técnicas de PLN. As técnicas podem ser aplicada em qualquer córpus obtido a partir de uma **API** ou a partir de uma página Web.

O **LLM** e a **API** selecionados devem ser informados na seguinte planilha:

> https://docs.google.com/spreadsheets/d/1iIUZcwnywO7RuF6VEJ8Rx9NDT1cwteyvsnkhYr0NWtU/edit?usp=sharing

>
As seguintes técnicas de PLN podem ser usadas:

*   Correção Gramatical
*   Classificação de Textos
*   Análise de Sentimentos
*   Detecção de Emoções
*   Extração de Palavras-chave
*   Tradução de Textos
*   Sumarização de Textos
*   Similaridade de Textos
*   Reconhecimento de Entidades Nomeadas
*   Sistemas de Perguntas e Respostas
>

**IMPORTANTE:** É obrigatório usar o e-mail da UFABC.


### **CRITÉRIOS DE AVALIAÇÃO**
---


Serão considerados como critérios de avaliação os seguintes pontos:

* Uso do framework **`LangChain`**.

* Escolha e uso de um **LLM**.

* Escolha e uso de uma **API** ou **Página Web**.

* Projeto disponível no Github.

* Apresentação (10 a 15 minutos).

* Criatividade no uso do framework **`LangChain`** em conjunto com o **LLM** e a **API**.



Também é interessante explorar técnicas como:
*   Geração de Dados Sintéticos
*   Geração Aumentada por Recuperação (RAG)
*   Agentes


**IMPORTANTE**: todo o código do notebook deve ser documentado e EXECUTADO. Código sem execução não será considerado.

### **IMPLEMENTAÇÃO**
---

### Visão geral
O usuário informa um **tema**. O notebook:
1. Busca o artigo mais relevante na **Wikipédia** via **MediaWiki Action API** (`list=search` + `prop=extracts`);
2. Usa **LangChain** + **Claude Haiku 5.5** para aplicar três técnicas de PLN sobre o texto do artigo:
   - **Sumarização de Textos**
   - **Reconhecimento de Entidades Nomeadas (NER)**, com saída estruturada (Pydantic)
   - **Sistema de Perguntas e Respostas** baseado no artigo (o modelo responde só com o que está no texto recuperado — uma forma simples de RAG)

Além das três técnicas, o notebook explora duas das técnicas sugeridas nos critérios de avaliação:
   - **Agentes:** um agente LangChain que decide sozinho quando pesquisar e ler artigos da Wikipédia para responder perguntas que exigem mais de um artigo;
   - **Geração de Dados Sintéticos:** o LLM cria um conjunto de perguntas com respostas de referência, usado para avaliar o sistema de perguntas e respostas.

### 1. Instalação das dependências

In [1]:
# langchain-anthropic: integração do LangChain com os modelos Claude
# requests: chamadas HTTP para a MediaWiki API
# python-dotenv: lê a chave de um arquivo .env quando o notebook roda fora do Colab
%pip install -q langchain langchain-anthropic requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 2. Configuração da chave da API da Anthropic
A chave **não** fica no código.
- **No Colab:** clique no ícone de chave (🔑 *Secrets*) na barra lateral, crie um segredo chamado `ANTHROPIC_API_KEY` com a sua chave do Claude Console e habilite o acesso do notebook.
- **Localmente (VS Code/Jupyter):** crie um arquivo `.env` na pasta do projeto com a linha `ANTHROPIC_API_KEY=...`. Ele está no `.gitignore` e não vai para o repositório.

In [2]:
import os

try:
    from google.colab import userdata  # rodando no Colab
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    from dotenv import load_dotenv  # rodando localmente
    load_dotenv()

print("Chave carregada:", bool(os.environ.get("ANTHROPIC_API_KEY")))

Chave carregada: True


### 3. Coleta do córpus: MediaWiki Action API
Duas chamadas ao endpoint `https://pt.wikipedia.org/w/api.php`:
- `action=query&list=search` → encontra o título do artigo mais relevante para o tema;
- `action=query&prop=extracts&explaintext=1` → baixa o conteúdo do artigo em texto puro.

A Wikimedia exige um cabeçalho `User-Agent` que identifique o cliente com um contato. A função `consultar_api` também guarda em cache as consultas já feitas e, se a API responder **429** (muitas requisições), espera o tempo indicado e tenta de novo.

In [3]:
import time

import requests

WIKI_API = "https://pt.wikipedia.org/w/api.php"
# A política de User-Agent da Wikimedia pede identificação do cliente com um contato
HEADERS = {"User-Agent": "PLN-UFABC-ProjetoPratico/1.0 (https://github.com/iantoniolo/pln-projeto-pratico; projeto academico)"}

_cache = {}  # evita repetir a mesma consulta (o mesmo artigo é usado em várias seções)


def consultar_api(params: dict, tentativas: int = 4) -> dict:
    """Chama a MediaWiki API e devolve o JSON; erros da API viram exceção com a mensagem original.

    Se a API responder 429 (muitas requisições), espera o tempo indicado em `Retry-After` e tenta de novo."""
    chave = tuple(sorted(params.items()))
    if chave in _cache:
        return _cache[chave]
    for tentativa in range(tentativas):
        resp = requests.get(WIKI_API, params={**params, "format": "json"}, headers=HEADERS, timeout=30)
        if resp.status_code != 429 or tentativa == tentativas - 1:
            break
        espera = int(resp.headers.get("Retry-After", 2 ** (tentativa + 1)))
        print(f"(MediaWiki API: limite de requisições atingido, nova tentativa em {espera}s)")
        time.sleep(espera)
    resp.raise_for_status()
    dados = resp.json()
    if "error" in dados:
        erro = dados["error"]
        raise RuntimeError(f"MediaWiki API: {erro.get('code')}: {erro.get('info')}")
    if "query" not in dados:
        raise RuntimeError(f"Resposta inesperada da MediaWiki API: {str(dados)[:500]}")
    _cache[chave] = dados["query"]
    return dados["query"]


def buscar_titulo(tema: str) -> str:
    """Retorna o título do artigo da Wikipédia mais relevante para o tema."""
    resultados = consultar_api({
        "action": "query",
        "list": "search",
        "srsearch": tema,
        "srlimit": 1,
    })["search"]
    if not resultados:
        raise ValueError(f"Nenhum artigo encontrado para: {tema}")
    return resultados[0]["title"]


def obter_texto(titulo: str, max_chars: int = 15000) -> str:
    """Baixa o texto puro do artigo (limitado a max_chars para controlar custo/tokens)."""
    paginas = consultar_api({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "titles": titulo,
        "redirects": 1,
    })["pages"]
    texto = next(iter(paginas.values())).get("extract", "")
    return texto[:max_chars]


# Teste rápido
titulo = buscar_titulo("Universidade Federal do ABC")
artigo = obter_texto(titulo)
print("Artigo:", titulo)
print("Tamanho:", len(artigo), "caracteres\n")
print(artigo[:800])

Artigo: Universidade Federal do ABC
Tamanho: 15000 caracteres

A Universidade Federal do ABC (UFABC) é uma instituição pública federal de ensino superior no ABC Paulista. Ocupa o 1° lugar entre as universidades brasileiras no Ranking SCImago nos quesitos “Excelência em Pesquisa”, “Publicações de alta qualidade” e “Impacto normalizado das suas publicações”. Foi avaliada pelo Índice Geral de Cursos (IGC) do MEC como a melhor universidade do Estado de São Paulo, sendo avaliada como a 1ª no ranking de cursos de graduação entre todas as universidades do Brasil. O IGC leva em consideração em sua avaliação fatores como infraestrutura, corpo docente, e nota dos formandos no ENADE. Ocupa o 1º lugar entre as universidades brasileiras no quesito "Internacionalização" no Ranking Universitário Folha 2019.
O presidente da comissão que formulou a proposta da univers


### 4. LLM via LangChain
`ChatAnthropic` é o wrapper do LangChain para os modelos Claude. O Claude Haiku 5.5 não aceita o parâmetro `temperature` (a API devolve erro 400), então usamos os valores padrão de amostragem. `max_tokens=4096` dá folga para o raciocínio interno do modelo, que conta dentro desse limite, e para a saída estruturada do NER.

In [4]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-haiku-5-5", max_tokens=4096)

# Teste de conexão
print(llm.invoke("Responda em uma frase: o que é PLN?").content)

PLN (Processamento de Linguagem Natural) é a área da inteligência artificial e da linguística computacional que permite que computadores compreendam, interpretem e gerem textos e falas humanas.


### 5. Técnica 1: Sumarização de Textos
Uma *chain* LCEL (`prompt | llm | parser`): o prompt recebe o artigo e pede um resumo em português, em tópicos.

In [5]:
prompt_resumo = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente que resume artigos enciclopédicos em português do Brasil, de forma fiel ao texto."),
    ("human", "Resuma o artigo abaixo em até 5 tópicos curtos.\n\nTítulo: {titulo}\n\nArtigo:\n{artigo}"),
])

chain_resumo = prompt_resumo | llm | StrOutputParser()

resumo = chain_resumo.invoke({"titulo": titulo, "artigo": artigo})
print(resumo)

- **Identidade:** A Universidade Federal do ABC (UFABC) é uma instituição pública federal de ensino superior, criada por lei de 2005, com campi em Santo André (sede) e São Bernardo do Campo.
- **Destaques:** Segundo o texto, a UFABC ficou em 1º lugar entre as universidades brasileiras no Ranking SCImago em pesquisa e publicações, foi considerada a melhor universidade paulista pelo IGC e está em 1º lugar em internacionalização no Ranking Universitário Folha 2019. É a única federal com 100% de docentes doutores.
- **Ensino:** Oferece bacharelados interdisciplinares em quadrimestres, com o Bacharelado em Ciência e Tecnologia (BC&T) e o Bacharelado em Ciências e Humanidades (BC&H). Após o ciclo básico, o aluno pode seguir para 21 cursos de graduação, incluindo engenharias, ou para o mestrado.
- **Expansão:** Em dezembro de 2024, foram aprovadas três licenciaturas (Pedagogia; Educação para as Infâncias, Linguagens e Artes; Geografia), e o curso de Ciência de Dados será ofertado a partir de 

### 6. Técnica 2: Reconhecimento de Entidades Nomeadas (NER)
Usamos `with_structured_output` com um esquema **Pydantic**: o LangChain força o modelo a devolver uma lista de entidades já tipadas (PESSOA, ORGANIZAÇÃO, LOCAL, DATA, OUTRO), pronta para virar tabela.

O prompt define cada tipo com exemplos (inclusive datas). Como o modelo nem sempre respeita uma quantidade pedida no prompt, a função `extrair_entidades` limita o resultado a 25 entidades no código, escolhendo em rodízio entre os tipos para que todos os tipos encontrados no texto apareçam na tabela.

In [6]:
from typing import List, Literal
from pydantic import BaseModel, Field
import pandas as pd


class Entidade(BaseModel):
    texto: str = Field(description="A entidade exatamente como aparece no texto")
    tipo: Literal["PESSOA", "ORGANIZACAO", "LOCAL", "DATA", "OUTRO"]


class ListaEntidades(BaseModel):
    entidades: List[Entidade] = Field(description="Entidades nomeadas distintas, ordenadas da mais relevante para a menos relevante")


prompt_ner = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um sistema de Reconhecimento de Entidades Nomeadas. Classifique cada entidade em um tipo:\n"
     "- PESSOA: nomes de pessoas\n"
     "- ORGANIZACAO: instituições, empresas, órgãos públicos, veículos de imprensa\n"
     "- LOCAL: cidades, estados, países, regiões, endereços\n"
     "- DATA: datas e anos citados no texto (ex.: \"2005\", \"26 de julho de 2005\")\n"
     "- OUTRO: demais nomes próprios (leis, prêmios, cursos, obras, produtos)\n"
     "Não repita entidades. Ordene a lista da entidade mais relevante para a menos relevante, "
     "e garanta que as primeiras posições cubram todos os tipos que aparecem no texto."),
    ("human", "Extraia as entidades nomeadas do texto abaixo.\n\n{artigo}"),
])

chain_ner = prompt_ner | llm.with_structured_output(ListaEntidades)


def extrair_entidades(artigo: str, max_entidades: int = 25) -> list[Entidade]:
    """Roda a chain de NER e devolve até `max_entidades` entidades, equilibradas entre os tipos.

    O limite é aplicado no código porque o modelo nem sempre respeita uma quantidade pedida no prompt.
    A seleção é feita em rodízio entre os tipos (mantendo a ordem de relevância do modelo dentro de
    cada tipo), para que nenhum tipo encontrado no texto fique de fora."""
    todas = chain_ner.invoke({"artigo": artigo}).entidades
    por_tipo = {}
    for e in todas:
        por_tipo.setdefault(e.tipo, []).append(e)
    selecionadas = []
    while len(selecionadas) < max_entidades and any(por_tipo.values()):
        for fila in por_tipo.values():
            if fila and len(selecionadas) < max_entidades:
                selecionadas.append(fila.pop(0))
    return selecionadas


entidades = extrair_entidades(artigo)
df_entidades = pd.DataFrame([e.model_dump() for e in entidades])
df_entidades.sort_values("tipo").reset_index(drop=True)

,texto,tipo
0,2005,DATA
1,2008,DATA
2,2004,DATA
3,2007,DATA
4,2009,DATA
5,ABC Paulista,LOCAL
6,Santo André,LOCAL
7,Brasil,LOCAL
8,Grande ABC,LOCAL
9,São Paulo,LOCAL


In [7]:
# Contagem de entidades por tipo
df_entidades["tipo"].value_counts()

tipo
ORGANIZACAO    5
PESSOA         5
LOCAL          5
DATA           5
OUTRO          5
Name: count, dtype: int64

### 7. Técnica 3: Sistema de Perguntas e Respostas
O artigo recuperado da API é injetado no prompt como **contexto**, e o modelo é instruído a responder **apenas** com base nele (Geração Aumentada por Recuperação simples). Se a resposta não estiver no artigo, ele deve dizer isso em vez de inventar.

In [8]:
prompt_qa = ChatPromptTemplate.from_messages([
    ("system",
     "Responda à pergunta usando SOMENTE o contexto fornecido (um artigo da Wikipédia). "
     "Se a resposta não estiver no contexto, responda: 'O artigo não traz essa informação.' "
     "Responda em português, de forma direta."),
    ("human", "Contexto:\n{artigo}\n\nPergunta: {pergunta}"),
])

chain_qa = prompt_qa | llm | StrOutputParser()

perguntas = [
    "Em que ano a universidade foi criada?",
    "Em quais cidades ficam os campi?",
    "Quem ganhou a Copa do Mundo de 2002?",  # fora do contexto: deve recusar
]

for p in perguntas:
    print("P:", p)
    print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")

P: Em que ano a universidade foi criada?


R: A UFABC foi criada em **2005**, pela Lei nº 11.145, datada de 26 de julho de 2005 e publicada no Diário Oficial da União em 27 de julho do mesmo ano. O projeto de lei foi encaminhado ao Congresso em 2004. 

P: Em quais cidades ficam os campi?


R: Os campi da UFABC estão sediados nos municípios de Santo André (sede) e São Bernardo do Campo. 

P: Quem ganhou a Copa do Mundo de 2002?


R: O artigo não traz essa informação. 



### 8. Pipeline completo: qualquer tema
Junta tudo em uma função: tema → Wikipédia → resumo + entidades + respostas.

In [9]:
def analisar_tema(tema: str, perguntas: list[str]):
    titulo = buscar_titulo(tema)
    artigo = obter_texto(titulo)
    print(f"=== Artigo: {titulo} ({len(artigo)} caracteres) ===\n")

    print("--- RESUMO ---")
    print(chain_resumo.invoke({"titulo": titulo, "artigo": artigo}), "\n")

    print("--- ENTIDADES NOMEADAS ---")
    ents = extrair_entidades(artigo)
    display(pd.DataFrame([e.model_dump() for e in ents]))

    print("\n--- PERGUNTAS E RESPOSTAS ---")
    for p in perguntas:
        print("P:", p)
        print("R:", chain_qa.invoke({"artigo": artigo, "pergunta": p}), "\n")


analisar_tema("Alan Turing", [
    "Onde Alan Turing nasceu?",
    "Qual foi a contribuição dele durante a Segunda Guerra Mundial?",
])

=== Artigo: Alan Turing (15000 caracteres) ===

--- RESUMO ---


- **Quem foi:** Matemático, lógico, criptoanalista e cientista da computação britânico (Londres, 1912 – Wilmslow, 1954), considerado o pai da ciência da computação teórica e da inteligência artificial.
- **Contribuição teórica:** Formalizou os conceitos de algoritmo e computação com a máquina de Turing, modelo de computador de uso geral. Em 1936, publicou o artigo "Sobre números computáveis".
- **Segunda Guerra Mundial:** Trabalhou em Bletchley Park, liderou a Hut 8 (análise naval alemã) e desenvolveu técnicas e uma máquina eletromecânica para quebrar a Enigma. Estima-se que seu trabalho encurtou a guerra na Europa em mais de dois anos e salvou mais de 14 milhões de vidas.
- **Pós-guerra:** Projetou o Mecanismo de Computação Automática no Laboratório Nacional de Física, ajudou a desenvolver os computadores de Manchester e se dedicou à biologia matemática, prevendo reações químicas oscilantes.
- **Perseguição e reconhecimento:** Foi processado em 1952 por atos homossexuais e aceitou cas

,texto,tipo
0,Alan Mathison Turing,PESSOA
1,Londres,LOCAL
2,23 de junho de 1912,DATA
3,Lei de Segredos Oficiais,OUTRO
4,Escola de Código e Cifras do Governo (GC&CS),ORGANIZACAO
5,Max Newman,PESSOA
6,Wilmslow,LOCAL
7,7 de junho de 1954,DATA
8,Enigma,OUTRO
9,Hut 8,ORGANIZACAO



--- PERGUNTAS E RESPOSTAS ---
P: Onde Alan Turing nasceu?


R: Alan Turing nasceu em Maida Vale, Londres, em 23 de junho de 1912. 

P: Qual foi a contribuição dele durante a Segunda Guerra Mundial?


R: Durante a Segunda Guerra Mundial, Turing trabalhou em Bletchley Park, centro britânico de criptoanálise. Ele liderou a Hut 8, responsável pela análise criptográfica naval alemã, e desenvolveu técnicas para acelerar a quebra das cifras alemãs. Entre elas estavam melhorias no método polonês de bombardeio e uma máquina eletromecânica capaz de encontrar configurações da Enigma. Seu trabalho foi crucial para decifrar mensagens interceptadas, o que ajudou os Aliados a vencer batalhas importantes, como a Batalha do Atlântico. Estima-se que isso encurtou a guerra na Europa em mais de dois anos e salvou mais de 14 milhões de vidas. 



### 9. Agente: o LLM decide quais buscas fazer
Nas seções anteriores, o **código** define a ordem dos passos (buscar → resumir → extrair → responder). Num **agente**, quem decide é o **LLM**: ele recebe ferramentas e escolhe quais chamar, com quais argumentos e quantas vezes, até ter informação suficiente para responder.

As funções da MediaWiki API viram duas **ferramentas** (`@tool`) do LangChain:
- `pesquisar_wikipedia`: busca artigos e devolve até 5 títulos com um trecho de cada;
- `ler_artigo`: baixa o texto de um artigo pelo título.

O `create_agent` do LangChain monta o laço: o modelo pede uma ferramenta → o código executa → o resultado volta para o modelo → repete até a resposta final. A função `perguntar_ao_agente` mostra cada ferramenta chamada, para deixar visível o raciocínio do agente.

Os exemplos exigem mais de um artigo: uma **comparação** entre duas pessoas e uma pergunta de **múltiplos saltos** (primeiro descobrir a cidade-sede da UFABC, depois buscar a fundação dessa cidade).

In [10]:
import html
import re

from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def pesquisar_wikipedia(consulta: str) -> str:
    """Pesquisa artigos na Wikipédia em português e devolve até 5 títulos com um trecho de cada.
    Use para descobrir o título exato de um artigo antes de lê-lo."""
    resultados = consultar_api({
        "action": "query",
        "list": "search",
        "srsearch": consulta,
        "srlimit": 5,
    })["search"]
    if not resultados:
        return "Nenhum artigo encontrado."
    linhas = []
    for r in resultados:
        trecho = html.unescape(re.sub(r"<[^>]+>", "", r["snippet"]))  # remove as marcações HTML do trecho
        linhas.append(f"- {r['title']}: {trecho}")
    return "\n".join(linhas)


@tool
def ler_artigo(titulo: str) -> str:
    """Lê o texto de um artigo da Wikipédia em português pelo título exato (até 8.000 caracteres)."""
    texto = obter_texto(titulo, max_chars=8000)
    return texto or f"Artigo '{titulo}' não encontrado."


agente = create_agent(
    model=llm,
    tools=[pesquisar_wikipedia, ler_artigo],
    system_prompt=(
        "Você é um assistente de pesquisa que responde em português do Brasil usando a Wikipédia. "
        "Use as ferramentas para pesquisar e ler os artigos necessários antes de responder. "
        "Baseie a resposta apenas no que leu e, no final, liste os artigos consultados."
    ),
)

print("Agente criado com as ferramentas:", [t.name for t in (pesquisar_wikipedia, ler_artigo)])


def perguntar_ao_agente(pergunta: str) -> str:
    """Executa o agente e mostra cada ferramenta que ele decidiu chamar."""
    print("PERGUNTA:", pergunta, "\n")
    resultado = agente.invoke({"messages": [{"role": "user", "content": pergunta}]})
    print("--- Ferramentas chamadas pelo agente ---")
    for msg in resultado["messages"]:
        for chamada in getattr(msg, "tool_calls", None) or []:
            print(f"🔧 {chamada['name']}({chamada['args']})")
    resposta = resultado["messages"][-1].text
    print("\n--- Resposta ---")
    print(resposta)
    return resposta

Agente criado com as ferramentas: ['pesquisar_wikipedia', 'ler_artigo']


In [11]:
perguntar_ao_agente("Compare as contribuições de Alan Turing e de John von Neumann para a computação.");

PERGUNTA: Compare as contribuições de Alan Turing e de John von Neumann para a computação. 



--- Ferramentas chamadas pelo agente ---
🔧 pesquisar_wikipedia({'consulta': 'Alan Turing'})
🔧 pesquisar_wikipedia({'consulta': 'John von Neumann'})
🔧 ler_artigo({'titulo': 'Alan Turing'})
🔧 ler_artigo({'titulo': 'John von Neumann'})
🔧 ler_artigo({'titulo': 'Máquina de Turing'})
🔧 ler_artigo({'titulo': 'Arquitetura de von Neumann'})

--- Resposta ---
## Alan Turing e John von Neumann na computação

As contribuições se complementam: Turing estabeleceu o que é computável, e von Neumann organizou a forma como um computador real é construído.

**Alan Turing: fundamentos teóricos e primeiros projetos**
- **Modelo teórico de computação:** a máquina de Turing, descrita no artigo de 1936, é um modelo abstrato que se restringe aos aspectos lógicos do funcionamento (memória, estados e transições). Turing a chamou de "máquina de computação universal", e ela pode modelar qualquer computador digital. O artigo também tratou do Entscheidungsproblem.
- **Formalização de algoritmo e computação:** segund

In [12]:
perguntar_ao_agente("Em que ano foi fundada a cidade onde fica a sede da Universidade Federal do ABC?");

PERGUNTA: Em que ano foi fundada a cidade onde fica a sede da Universidade Federal do ABC? 



--- Ferramentas chamadas pelo agente ---
🔧 pesquisar_wikipedia({'consulta': 'Universidade Federal do ABC'})
🔧 ler_artigo({'titulo': 'Universidade Federal do ABC'})
🔧 ler_artigo({'titulo': 'Santo André (São Paulo)'})

--- Resposta ---
Não há um ano de fundação único na Wikipédia. A sede da UFABC fica em **Santo André (SP)**, e o artigo sobre a cidade traz estas datas:

- **Origem colonial:** o povoado Santo André da Borda do Campo foi fundado por João Ramalho em algum momento entre o início do século XVI e 1532.
- **Elevação a vila:** em 8 de abril de 1553, o povoado foi elevado a vila pelo governador-geral Tomé de Sousa.
- **Transferência:** em 1560, a vila foi transferida para os campos de Piratininga, por ordem de Mem de Sá.
- **Município atual:** em 1910 foi criado o Distrito de Paz de Santo André, dentro do município de São Bernardo. Em 1938, o distrito passou a ser a sede municipal, e o nome da cidade passou a ser Santo André.

Se você quer o ano da fundação do povoado original, a

### 10. Geração de Dados Sintéticos: avaliação do sistema de perguntas e respostas
Para medir a qualidade do sistema de perguntas e respostas (seção 7), precisamos de perguntas com **respostas de referência**. Em vez de escrevê-las à mão, o próprio LLM gera um **conjunto sintético** a partir do artigo da UFABC, com saída estruturada (Pydantic).

Depois, cada pergunta passa pelo `chain_qa`, e um terceiro prompt atua como **juiz** (*LLM-as-a-judge*): compara a resposta do sistema com a referência e decide se estão corretas, com uma justificativa. O resultado é uma **acurácia** do sistema.

In [13]:
class ParPerguntaResposta(BaseModel):
    pergunta: str = Field(description="Pergunta objetiva que pode ser respondida apenas com o artigo")
    resposta_referencia: str = Field(description="Resposta correta e curta, retirada do artigo")


class ConjuntoSintetico(BaseModel):
    itens: List[ParPerguntaResposta] = Field(description="Pares de pergunta e resposta sobre o artigo")


prompt_sintetico = ChatPromptTemplate.from_messages([
    ("system",
     "Você cria conjuntos de avaliação para sistemas de perguntas e respostas. "
     "Cada pergunta deve ter uma única resposta correta, presente de forma explícita no artigo."),
    ("human",
     "Crie {n} pares de pergunta e resposta sobre o artigo abaixo. Varie os tipos: datas, pessoas, lugares, números e fatos.\n\n"
     "Artigo:\n{artigo}"),
])

chain_sintetico = prompt_sintetico | llm.with_structured_output(ConjuntoSintetico)

N_PERGUNTAS = 8
dados_sinteticos = chain_sintetico.invoke({"n": N_PERGUNTAS, "artigo": artigo}).itens[:N_PERGUNTAS]

pd.set_option("display.max_colwidth", 200)
df_sintetico = pd.DataFrame([d.model_dump() for d in dados_sinteticos])
df_sintetico

,pergunta,resposta_referencia
0,"Em que ano o Ministério da Educação encaminhou ao Congresso Nacional o Projeto de Lei 3.962/2004, que previa a criação da Fundação Universidade Federal do ABC?",2004
1,Qual é o número da lei sancionada que criou a UFABC?,11.145
2,Quem foi o primeiro reitor da UFABC?,Hermano Tavares
3,Quem foi nomeado reitor da UFABC em janeiro de 2010?,Helio Waldman
4,Em que município está localizada a sede da UFABC?,Santo André
5,Quantos quadrimestres por ano os alunos dos bacharelados interdisciplinares da UFABC estudam?,3
6,"Em março de 2007, quantos dos 11 pedidos de abertura de cursos de pós-graduação stricto sensu a CAPES aprovou?",10
7,Quantos eixos principais possui o Bacharelado em Ciência e Tecnologia (BC&T)?,6


In [14]:
class Avaliacao(BaseModel):
    correta: bool = Field(description="True se a resposta do sistema tem o mesmo sentido da resposta de referência")
    justificativa: str = Field(description="Explicação curta da decisão")


prompt_juiz = ChatPromptTemplate.from_messages([
    ("system",
     "Você avalia respostas de um sistema de perguntas e respostas. Considere a resposta correta "
     "se ela contém a informação da referência, mesmo com outras palavras ou com detalhes a mais que não a contradigam."),
    ("human", "Pergunta: {pergunta}\nResposta de referência: {referencia}\nResposta do sistema: {resposta}"),
])

chain_juiz = prompt_juiz | llm.with_structured_output(Avaliacao)

linhas = []
for item in dados_sinteticos:
    resposta = chain_qa.invoke({"artigo": artigo, "pergunta": item.pergunta})
    avaliacao = chain_juiz.invoke({
        "pergunta": item.pergunta,
        "referencia": item.resposta_referencia,
        "resposta": resposta,
    })
    linhas.append({
        "pergunta": item.pergunta,
        "referencia": item.resposta_referencia,
        "resposta_do_sistema": resposta,
        "correta": avaliacao.correta,
        "justificativa": avaliacao.justificativa,
    })

df_avaliacao = pd.DataFrame(linhas)
print(f"Acurácia do sistema de perguntas e respostas: {df_avaliacao['correta'].mean():.0%} "
      f"({df_avaliacao['correta'].sum()} de {len(df_avaliacao)})")
df_avaliacao

Acurácia do sistema de perguntas e respostas: 100% (8 de 8)


,pergunta,referencia,resposta_do_sistema,correta,justificativa
0,"Em que ano o Ministério da Educação encaminhou ao Congresso Nacional o Projeto de Lei 3.962/2004, que previa a criação da Fundação Universidade Federal do ABC?",2004,O Ministério da Educação encaminhou o Projeto de Lei 3.962/2004 ao Congresso Nacional no ano de 2004.,True,"A resposta do sistema indica o ano de 2004, que é o mesmo da referência, apenas com palavras diferentes e sem contradição."
1,Qual é o número da lei sancionada que criou a UFABC?,11.145,"A lei que criou a UFABC é a de número **11.145**, sancionada pelo presidente Luiz Inácio Lula da Silva e publicada no Diário Oficial da União de 27 de julho de 2005 (datada de 26 de julho de 2005).",True,"A resposta do sistema indica a lei nº 11.145, que é exatamente a referência. Os detalhes adicionais (sanção por Lula, publicação no DOU e datas) não contradizem a referência e são consistentes com..."
2,Quem foi o primeiro reitor da UFABC?,Hermano Tavares,"O primeiro reitor da UFABC foi o professor Hermano Tavares, ex-reitor da UNICAMP, cujo mandato durou de 2005 até 2007.",True,"A resposta do sistema identifica Hermano Tavares como primeiro reitor da UFABC, igual à referência. Os detalhes adicionais (ex-reitor da UNICAMP, mandato 2005–2007) não contradizem a referência, e..."
3,Quem foi nomeado reitor da UFABC em janeiro de 2010?,Helio Waldman,"Em janeiro de 2010, o presidente Lula nomeou o professor Helio Waldman para exercer o cargo de reitor da UFABC nos quatro anos seguintes.",True,"A resposta do sistema identifica Helio Waldman como o nomeado reitor da UFABC em janeiro de 2010, que é exatamente a informação da referência. O detalhe adicional (nomeação pelo presidente Lula e ..."
4,Em que município está localizada a sede da UFABC?,Santo André,Santo André.,True,"A resposta do sistema, ""Santo André"", é idêntica à referência. A diferença está apenas no ponto final, o que não altera o sentido."
5,Quantos quadrimestres por ano os alunos dos bacharelados interdisciplinares da UFABC estudam?,3,Os alunos estudam 3 quadrimestres por ano (3 períodos de 4 meses cada).,True,"A resposta do sistema afirma 3 quadrimestres por ano, o que coincide com a referência. O detalhe adicional (3 períodos de 4 meses cada) é consistente com a definição de quadrimestre e não contradi..."
6,"Em março de 2007, quantos dos 11 pedidos de abertura de cursos de pós-graduação stricto sensu a CAPES aprovou?",10,Dez dos 11 pedidos.,True,"A resposta do sistema ""Dez dos 11 pedidos"" tem o mesmo sentido da referência (10). Apenas expressa o número por extenso e com contexto adicional, sem contradição."
7,Quantos eixos principais possui o Bacharelado em Ciência e Tecnologia (BC&T)?,6,O Bacharelado em Ciência e Tecnologia (BC&T) oferece disciplinas em torno de 6 eixos principais.,True,"A resposta do sistema indica 6 eixos principais, o mesmo valor da referência. O texto adicional sobre as disciplinas não contradiz a referência."


**Limitações desta avaliação:** o mesmo modelo gera as perguntas, responde e julga, o que tende a deixar o resultado otimista. As perguntas saem do mesmo trecho que o sistema lê, e o conjunto é pequeno. Numa avaliação mais rigorosa, as referências seriam revisadas por pessoas e o juiz seria um modelo diferente.

### 11. Conclusão
- **API:** a MediaWiki Action API forneceu o córpus (artigos da Wikipédia em português) sob demanda, a partir de qualquer tema.
- **LangChain:** usado para montar as *chains* (prompt → LLM → parser), obter saída estruturada com Pydantic e criar um agente com ferramentas.
- **LLM:** o Claude Haiku 5.5 aplicou três técnicas de PLN: sumarização, NER e perguntas e respostas fundamentadas no texto recuperado (RAG simples).
- **Agente:** o modelo decidiu sozinho quais artigos pesquisar e ler para responder perguntas de comparação e de múltiplos saltos.
- **Dados sintéticos:** o LLM gerou um conjunto de avaliação, e um juiz automático mediu a acurácia do sistema de perguntas e respostas.